# NB00 — setup + tests (CPU)

Clones the `eegrep` code, installs it, runs the full pytest suite and the synthetic end-to-end smoke run (no real data), and records the environment for the reproducibility statement.

Settings: **Accelerator: None (CPU)** · **Internet: On**.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
sha = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
print('git sha', sha)

In [ ]:
sh(f'pip install -q -e "{SRC}[dev]"', log=f'{WORK}/pip_install.log')

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider', log=f'{WORK}/pytest.txt')

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m eegrep smoke --workdir /tmp/eegrep-smoke', log=f'{WORK}/smoke.txt')
shutil.copy('/tmp/eegrep-smoke/runs.jsonl', f'{WORK}/smoke_runs.jsonl')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': sha, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown'),
       'kernel_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)